# Ingestão Raw: Dimensões

**Objetivo:** ler os arquivos JSON das 3 dimensões (Produtos, Lojas, Representantes) da Landing Zone, usando Databricks Autoloader, e gravar como tabelas Delta na camada Raw — mantendo o dado fiel à origem (schema-on-read), sem qualquer tratamento ou tipagem além do necessário para persistência.

**Fonte:** `/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/{produtos,lojas,representantes}/`

**Destino:** tabelas Delta `raw.produtos`, `raw.lojas`, `raw.representantes` (schema `raw`, catalog `poc_latam_food`).

**Modo de execução:** batch (`trigger(availableNow=True)`) — o Autoloader processa todos os arquivos novos disponíveis no momento e encerra, em vez de ficar continuamente "escutando" (streaming contínuo), alinhado à nossa decisão de orquestração via job diário, não streaming.

**Checkpoint:** `/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/dimensoes/` — pasta de controle do Autoloader, que rastreia quais arquivos já foram processados, evitando reprocessamento duplicado.

**Observação de retenção:** diferente da tabela de Vendas, as dimensões **não estão sujeitas ao TTL de 48h** — por serem tabelas de baixo volume, carregadas uma única vez, sua retenção na Raw é permanente, servindo inclusive como cópia de auditoria da carga original.

In [0]:
# imports e criação do schema

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.raw")

print("Schema 'raw' verificado/criado com sucesso.")

In [0]:
# Ingestão produtos

caminho_origem_produtos = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/produtos"
caminho_checkpoint_produtos = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/dimensoes/produtos"
caminho_schema_produtos = "/Volumes/poc_latam_food/landing/blob_simulado/_schemas/dimensoes/produtos"

from pyspark.sql.functions import current_timestamp

df_stream_produtos = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", caminho_schema_produtos)
    .load(caminho_origem_produtos)
    .withColumn("data_ingestao", current_timestamp())
)

(
    df_stream_produtos
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_produtos)
    .trigger(availableNow=True)
    .toTable("poc_latam_food.raw.produtos")
)

print("Ingestão de Produtos concluída.")

In [0]:
# Ingestão lojas

caminho_origem_lojas = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/lojas"
caminho_checkpoint_lojas = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/dimensoes/lojas"
caminho_schema_lojas = "/Volumes/poc_latam_food/landing/blob_simulado/_schemas/dimensoes/lojas"

df_stream_lojas = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", caminho_schema_lojas)
    .load(caminho_origem_lojas)
    .withColumn("data_ingestao", current_timestamp())
)

(
    df_stream_lojas
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_lojas)
    .trigger(availableNow=True)
    .toTable("poc_latam_food.raw.lojas")
)

print("Ingestão de Lojas concluída.")

In [0]:
# Ingestão representantes

caminho_origem_representantes = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/representantes"
caminho_checkpoint_representantes = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/dimensoes/representantes"
caminho_schema_representantes = "/Volumes/poc_latam_food/landing/blob_simulado/_schemas/dimensoes/representantes"

df_stream_representantes = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", caminho_schema_representantes)
    .load(caminho_origem_representantes)
    .withColumn("data_ingestao", current_timestamp())
)

(
    df_stream_representantes
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_representantes)
    .trigger(availableNow=True)
    .toTable("poc_latam_food.raw.representantes")
)

print("Ingestão de Representantes concluída.")

In [0]:
# Validação visual - conferência das tabelas Raw

print("=== raw.produtos ===")
spark.table("poc_latam_food.raw.produtos").display()

print("=== raw.lojas ===")
spark.table("poc_latam_food.raw.lojas").display()

print("=== raw.representantes ===")
spark.table("poc_latam_food.raw.representantes").display()

print("Contagens:")
print(f"Produtos: {spark.table('poc_latam_food.raw.produtos').count()}")
print(f"Lojas: {spark.table('poc_latam_food.raw.lojas').count()}")
print(f"Representantes: {spark.table('poc_latam_food.raw.representantes').count()}")